In [ ]:
# STEP 0: Mount Google Drive
import os
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
    project_path = '/content/gdrive/MyDrive/feral-cat-fcm'
except ImportError:
    project_path = os.path.abspath('feral-cat-fcm')  # running locally

# Define paths
auto_crops = f'{project_path}/outputs/crops/auto'
manual_crops = f'{project_path}/outputs/crops/manual'
work_path = f'{project_path}/outputs/fcm_work'
fcm_path = f'{project_path}/outputs/fcm_datasets'
body_parts = ['back-leg', 'body', 'front-leg', 'tail']

In [ ]:
# Functions
import random
import re
import shutil
from pathlib import Path

splits = ['train', 'val', 'test']


def images(path):
    path = Path(path)
    return sorted(p for p in path.iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png')) if path.is_dir() else []


def subfolders(path):
    path = Path(path)
    return sorted(d.name for d in path.iterdir() if d.is_dir()) if path.is_dir() else []


def new_folder(path):
    if Path(path).exists():
        raise FileExistsError(f'{path} already exists, delete it first')
    Path(path).mkdir(parents=True)


def copy_images(files, dest):
    Path(dest).mkdir(parents=True, exist_ok=True)
    for f in files:
        shutil.copy(f, Path(dest, f.name))


def counts(dataset, split):
    table = {}
    for part in body_parts:
        for cat in subfolders(Path(dataset, part, split)):
            table.setdefault(cat, {})[part] = len(images(Path(dataset, part, split, cat)))
    return table


# Balance: every body part of a cat is cut down to the count of its rarest body part
def balance(src, dst):
    rng = random.Random(123)
    new_folder(dst)
    for cat in sorted(set().union(*(subfolders(Path(src, part)) for part in body_parts))):
        per_part = {part: images(Path(src, part, cat)) for part in body_parts}
        n = min(len(files) for files in per_part.values())
        if n == 0:
            print(f'{cat}: skipped, a body part has no images')
            continue
        for part, files in per_part.items():
            copy_images(sorted(rng.sample(files, n)), Path(dst, part, cat))
        print(f'{cat}: {n} images per body part')


# Split each cat, with at least 1 image in val and test
def split(src, dst, ratios=(0.70, 0.15, 0.15)):
    new_folder(dst)
    for part in body_parts:
        rng = random.Random(123)
        for cat in subfolders(Path(src, body_parts[0])):
            files = images(Path(src, part, cat))
            rng.shuffle(files)
            n = len(files)
            n_val = max(1, int(n * ratios[1])) if n >= 3 else 0
            n_test = max(1, int(n * ratios[2])) if n >= 3 and ratios[2] > 0 else 0
            n_train = n - n_val - n_test
            copy_images(files[:n_train], Path(dst, part, 'train', cat))
            copy_images(files[n_train:n_train + n_val], Path(dst, part, 'val', cat))
            if ratios[2] > 0:
                copy_images(files[n_train + n_val:], Path(dst, part, 'test', cat))


# Images per cat in each split, and whether all body parts match
def check(dataset):
    folders = [Path(dataset, part, s) for part in body_parts for s in splits if Path(dataset, part, s).is_dir()]
    all_cats = set().union(*(subfolders(f) for f in folders))
    for f in folders:
        for cat in all_cats:
            Path(f, cat).mkdir(exist_ok=True)  # same cats everywhere so the labels line up in the FCM

    balanced = True
    print('cat'.ljust(28) + ''.join(s.rjust(7) for s in splits))
    for cat in sorted(all_cats):
        row = ''
        for s in splits:
            c = counts(dataset, s).get(cat, {})
            balanced = balanced and len(set(c.values())) <= 1
            row += str(max(c.values(), default='-')).rjust(7)
        print(cat.ljust(28) + row)
    print('balanced' if balanced else 'NOT balanced, run rebalance()')


# Cut body parts back to the same count after fixing crops by hand
def rebalance(dataset, split_names=('train', 'val')):
    rng = random.Random(123)
    for s in split_names:
        for cat, c in counts(dataset, s).items():
            n = min(c.get(part, 0) for part in body_parts)
            for part in body_parts:
                files = images(Path(dataset, part, s, cat))
                for f in rng.sample(files, len(files) - n):
                    f.unlink()


def copy_dataset(src, dst):
    new_folder(dst)
    shutil.copytree(src, dst, dirs_exist_ok=True)


def copy_test_set(src, dst):
    for part in body_parts:
        shutil.copytree(Path(src, part, 'test'), Path(dst, part, 'test'), dirs_exist_ok=True)


# Leave out manual crops from the same image and body part as a test crop (data leakage)
def source_image(f):
    return re.sub(r'_crop_\d+_cls\d+.*$', '', Path(f).stem)


def remove_test_overlap(src, test_dataset, dst):
    new_folder(dst)
    removed = 0
    for part in body_parts:
        test_set = Path(test_dataset, part, 'test')
        test_images = {source_image(f) for cat in subfolders(test_set) for f in images(test_set / cat)}
        for cat in subfolders(Path(src, part)):
            files = images(Path(src, part, cat))
            keep = [f for f in files if source_image(f) not in test_images]
            removed += len(files) - len(keep)
            copy_images(keep, Path(dst, part, cat))
    print(f'Removed {removed} manual crops that overlap with the test set')


# Cut a dataset down to the same number of images per cat as a reference dataset
def match_size(dataset, reference, split_names=('train', 'val')):
    rng = random.Random(123)
    for s in split_names:
        ref = counts(reference, s)
        for cat in counts(dataset, s):
            n = min(ref.get(cat, {}).values(), default=0)
            for part in body_parts:
                files = images(Path(dataset, part, s, cat))
                for f in rng.sample(files, max(0, len(files) - n)):
                    f.unlink()


def merge(datasets, dst):
    new_folder(dst)
    for dataset in datasets:
        for part in body_parts:
            for s in ('train', 'val'):
                for cat in subfolders(Path(dataset, part, s)):
                    copy_images(images(Path(dataset, part, s, cat)), Path(dst, part, s, cat))

In [ ]:
# STEP 1: Test 1, balance the automatic crops and split 70/15/15
balance(auto_crops, f'{work_path}/auto_balanced')
split(f'{work_path}/auto_balanced', f'{fcm_path}/test1')
check(f'{fcm_path}/test1')

In [ ]:
# STEP 2: Sanitise the test set by hand
# Check every crop in test1/<body part>/test is the right body part. This test set is used for every test.
check(f'{fcm_path}/test1')

In [ ]:
# STEP 3: Test 2, copy Test 1
copy_dataset(f'{fcm_path}/test1', f'{fcm_path}/test2')

In [ ]:
# STEP 4: Sanitise train and val by hand, then rebalance
# In test2/<body part>/train and val: move crops of the wrong body part to the right folder,
# delete crops that are not a body part or are a duplicate. Don't change the test folders.
rebalance(f'{fcm_path}/test2')
check(f'{fcm_path}/test2')

In [ ]:
# STEP 5: Test 3a, manual crops without the test set overlap, split 80/20, same test set
remove_test_overlap(manual_crops, f'{fcm_path}/test1', f'{work_path}/manual_no_overlap')
balance(f'{work_path}/manual_no_overlap', f'{work_path}/manual_balanced')
split(f'{work_path}/manual_balanced', f'{fcm_path}/test3a', ratios=(0.80, 0.20, 0))
copy_test_set(f'{fcm_path}/test1', f'{fcm_path}/test3a')
check(f'{fcm_path}/test3a')

In [ ]:
# STEP 6: Test 3b, Test 3a cut down to the size of Test 2
copy_dataset(f'{fcm_path}/test3a', f'{fcm_path}/test3b')
match_size(f'{fcm_path}/test3b', f'{fcm_path}/test2')
check(f'{fcm_path}/test3b')

In [ ]:
# STEP 7: Test 4, Test 3a + Test 2, same test set
merge([f'{fcm_path}/test3a', f'{fcm_path}/test2'], f'{fcm_path}/test4')
copy_test_set(f'{fcm_path}/test1', f'{fcm_path}/test4')
check(f'{fcm_path}/test4')